In [1]:
!pip install transformers peft accelerate bitsandbytes
!pip install trl
!pip install -U bitsandbytes>=0.46.1

  Using cached transformers-5.3.0-py3-none-any.whl.metadata (32 kB)
  Using cached peft-0.18.1-py3-none-any.whl.metadata (14 kB)
  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
  Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
  Using cached huggingface_hub-1.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached regex-2026.2.28-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
  Using cached typer-0.24.1-py3-none-any.whl.metadata (16 kB)
  Using cached safetensors-0.7.0-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.1 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached hf_xet-1.3.2-cp37-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (4.9 kB)
  Using cached click-8.3.1-py3-none-any.whl.metadata (2.6 kB)
  Using cache

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer
from transformers import TrainingArguments
import time

In [3]:
# --------------------------
# Device
# --------------------------
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("Device:", device)

Device: cuda


In [4]:
# --------------------------
# Dataset
# --------------------------
dataset = load_dataset(
    "FreedomIntelligence/medical-o1-reasoning-SFT",
    "zh",
    split="train[0:]",
) # using a small subset for testing — adjust as needed for your experiments
print(f"Loaded {len(dataset)} samples.")


README.md: 0.00B [00:00, ?B/s]

medical_o1_sft_Chinese.json:   0%|          | 0.00/50.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/20171 [00:00<?, ? examples/s]

Loaded 20171 samples.


In [5]:
# --------------------------
# Model selection
# --------------------------
from transformers import BitsAndBytesConfig # Import BitsAndBytesConfig

model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Define the quantization configuration
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=False,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config, # Pass the quantization config here
    device_map="auto", # Automatically map model layers to available devices
)

# model.to(device)    # No longer needed with device_map="auto"

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

AttributeError: 'Qwen2ForCausalLM' object has no attribute 'set_submodule'

In [ ]:
max_seq_length = 2048

def format_example(example):
    text = f"""### Question:
{example["Question"]}

### Response:
{example["Response"]}"""
    return {"text": text}

dataset = dataset.map(format_example)

def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=max_seq_length,
    )

dataset = dataset.map(tokenize_function, batched=True)
dataset = dataset.remove_columns(
    [col for col in dataset.column_names if col not in ["input_ids", "attention_mask", "labels"]]
)
dataset.set_format(type="torch", columns=["input_ids", "attention_mask"])
#dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])


In [ ]:
# Prepare trainer

# --------------------------
# LoRA config
# --------------------------
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
) # these target modules are a good starting point for many models,
  # but you may want to adjust based on your specific architecture


model = get_peft_model(model, lora_config)  # wrap the model with LoRA adapter

# --------------------------
# Training args
# --------------------------
training_args = TrainingArguments(
    output_dir="./outputs.tmp",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    num_train_epochs=1,
    learning_rate=1e-4,
    logging_steps=100,
    lr_scheduler_type="cosine",             # Smooth decay
    weight_decay=0.01,                      # Light regularization
    bf16=True,                 # If using torch (Apple Silicon supports bf16 well)
    max_steps= 100,
    #evaluation_strategy="steps",            # Or "epoch" if you have val set
    save_strategy="steps",     # important
    save_steps=1000,           # save every 1000 steps
    save_total_limit=3,        # keep last 3 checkpoints
    report_to="none",
  )


trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
)



In [ ]:
# --------------------------
# Train
# --------------------------
print("Start training")
start = time.perf_counter()
try:
    trainer.train()
except KeyboardInterrupt:
    print("Interrupted! Saving checkpoint...")
    trainer.save_model("./outputs/interrupted")

end = time.perf_counter()
print("End training.")

elapsed = end - start
print(f"Elapsed time: {elapsed:.3f} seconds")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Start training


Step,Training Loss
100,0.685685


End training.
Elapsed time: 686.545 seconds


In [ ]:
# --------------------------
# Save
# --------------------------
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")
